# CANSAT Duck2Dragon — Data Analysis

Reads `log.txt` from ground station serial logger (`read_serial.py`).

## CSV fields (23)
| # | Field | Unit | Source |
|---|-------|------|--------|
| 0 | millis | ms | ESP32 |
| 1 | lat | deg | GPS |
| 2 | lon | deg | GPS |
| 3 | alt_gps | m | GPS |
| 4 | sats | count | GPS |
| 5 | alt_baro | m | MS5611 |
| 6 | temp | °C | MS5611 |
| 7 | pressure | hPa | MS5611 |
| 8 | ax | m/s² | BNO085 linear accel |
| 9 | ay | m/s² | BNO085 linear accel |
| 10 | az | m/s² | BNO085 linear accel |
| 11 | gx | rad/s | BNO085 gyro |
| 12 | gy | rad/s | BNO085 gyro |
| 13 | gz | rad/s | BNO085 gyro |
| 14 | qw | — | BNO085 quaternion |
| 15 | qx | — | BNO085 quaternion |
| 16 | qy | — | BNO085 quaternion |
| 17 | qz | — | BNO085 quaternion |
| 18 | high_ax | g | ADXL375 |
| 19 | high_ay | g | ADXL375 |
| 20 | high_az | g | ADXL375 |
| 21 | voltage | V | INA219 |
| 22 | current | mA | INA219 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LOG_PATH = Path('log.txt')

COLUMNS = [
    'millis','lat','lon','alt_gps','sats',
    'alt_baro','temp','pressure',
    'ax','ay','az',
    'gx','gy','gz',
    'qw','qx','qy','qz',
    'high_ax','high_ay','high_az',
    'voltage','current'
]

rows = []
with open(LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split(',')
        if len(parts) == 23:
            rows.append(parts)

df = pd.DataFrame(rows, columns=COLUMNS)
df = df.apply(pd.to_numeric, errors='coerce')
df['time_s'] = df['millis'] / 1000.0

print(f'Loaded {len(df)} rows')
df.head()

In [ ]:
# Summary statistics
df.describe().round(3)

In [ ]:
# Data quality check
print('NaN count per column:')
print(df.isna().sum())
print(f'\nGPS fix rows (sats > 0): {(df.sats > 0).sum()} / {len(df)}')
print(f'Baro valid rows (alt_baro != 0): {(df.alt_baro != 0).sum()} / {len(df)}')
print(f'Duration: {df.time_s.max():.1f} s')
print(f'Sample rate: {len(df)/df.time_s.max():.2f} Hz')

In [ ]:
# Altitude: GPS vs Barometric
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.time_s, df.alt_baro, label='Baro (MS5611)', linewidth=1.2)
ax.plot(df.time_s, df.alt_gps,  label='GPS', linewidth=1.0, alpha=0.7)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Altitude (m)')
ax.set_title('Altitude — GPS vs Barometric')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature & Pressure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(df.time_s, df.temp, color='tomato', linewidth=1.2)
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('MS5611 Temperature')
ax1.grid(True, alpha=0.3)
ax2.plot(df.time_s, df.pressure, color='steelblue', linewidth=1.2)
ax2.set_ylabel('Pressure (hPa)')
ax2.set_xlabel('Time (s)')
ax2.set_title('MS5611 Pressure')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# BNO085 Linear Acceleration
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df.time_s, df.ax, label='ax')
axes[0].plot(df.time_s, df.ay, label='ay')
axes[0].plot(df.time_s, df.az, label='az')
axes[0].set_ylabel('m/s²')
axes[0].set_title('BNO085 Linear Acceleration')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

df['accel_mag'] = np.sqrt(df.ax**2 + df.ay**2 + df.az**2)
axes[1].plot(df.time_s, df.accel_mag, color='purple')
axes[1].set_ylabel('|a| (m/s²)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Total Acceleration Magnitude')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ADXL375 High-G
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.time_s, df.high_ax, label='high_ax')
ax.plot(df.time_s, df.high_ay, label='high_ay')
ax.plot(df.time_s, df.high_az, label='high_az')
ax.set_xlabel('Time (s)')
ax.set_ylabel('g')
ax.set_title('ADXL375 High-G Acceleration (±200g)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# BNO085 Gyroscope
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.time_s, df.gx, label='gx')
ax.plot(df.time_s, df.gy, label='gy')
ax.plot(df.time_s, df.gz, label='gz')
ax.set_xlabel('Time (s)')
ax.set_ylabel('rad/s')
ax.set_title('BNO085 Gyroscope')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Power: Voltage & Current
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax1.plot(df.time_s, df.voltage, color='gold', linewidth=1.2)
ax1.set_ylabel('Voltage (V)')
ax1.set_title('INA219 Bus Voltage')
ax1.grid(True, alpha=0.3)
ax2.plot(df.time_s, df.current, color='orange', linewidth=1.2)
ax2.set_ylabel('Current (mA)')
ax2.set_xlabel('Time (s)')
ax2.set_title('INA219 Current')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Avg voltage: {df.voltage.mean():.3f} V')
print(f'Avg current: {df.current.mean():.1f} mA')
print(f'Avg power:   {(df.voltage * df.current).mean():.1f} mW')

In [ ]:
# GPS Track
gps_valid = df[(df.sats > 0) & (df.lat != 0) & (df.lon != 0)]
if len(gps_valid) > 0:
    fig, ax = plt.subplots(figsize=(7, 7))
    sc = ax.scatter(gps_valid.lon, gps_valid.lat,
                    c=gps_valid.alt_gps, cmap='viridis', s=10)
    plt.colorbar(sc, ax=ax, label='Alt GPS (m)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'GPS Track ({len(gps_valid)} points)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No valid GPS fix in log.')

In [ ]:
# Apogee detection + vertical speed
idx_apogee = df.alt_baro.idxmax()
apogee_alt = df.alt_baro[idx_apogee]
apogee_t   = df.time_s[idx_apogee]
print(f'Apogee (baro): {apogee_alt:.2f} m at t={apogee_t:.1f} s')

df['alt_smooth'] = df.alt_baro.rolling(5, center=True).mean()
df['vz'] = df.alt_smooth.diff() / df.time_s.diff()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.time_s, df.vz, color='green', linewidth=1.0, label='Vz')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(apogee_t, color='red', linewidth=1.0, linestyle='--', label=f'Apogee t={apogee_t:.1f}s')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Vz (m/s)')
ax.set_title('Vertical Speed (from baro)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Export cleaned CSV
out_path = Path('log_clean.csv')
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows to {out_path}')